In [ ]:
from statistics import stdev

from PIL.features import features

from rl_trading_lab.environment.trading_env import TradingEnv, Action

In [ ]:
import logging

# Configure logging
logging.basicConfig(
  level=logging.DEBUG,
  format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

- Create a small pandas DataFrame suitable for initializing TradingEnv.
- Columns: timestamp, open, high, low, close, volume.
- Generate 200 rows of plausible OHLCV data with a datetime index.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

features = [
    "ratio_sma_5_close", "ratio_sma_20_close", "ratio_range_close", "fracdiff_0.4_zscore"
]
columns = ["timestamp", "close"] + features

df = pd.read_parquet("../sample_data/btcusdt_sample_10k.parquet",
                     columns=columns
                     )
df

In [ ]:
df.sort_values(by="timestamp", inplace=True)
df

In [ ]:
df.dropna(axis=0, inplace=True)

In [ ]:
def make_env():
    return TradingEnv(df,
                      lookback_window=20,
                      randomize_start=True,
                      one_trade_mode=True,
                      hold_closes_position=True,
                      reward_type="returns",
                      min_holding_period=2,
                      min_episode_length=2,
                      max_position_pct=0.2,
                      price_column="close",
                      discrete_actions=True,
                      commission_rate=0.0,
                      slippage_rate=0.0,
                      features_to_use=features,
                      )

In [ ]:
env = make_env()

In [ ]:
obs = env.reset()
obs

In [ ]:
obs, reward, terminated, truncated, info = env.step(action=Action.BUY)

In [ ]:
obs

In [ ]:
reward

In [ ]:
terminated

In [ ]:
info

In [ ]:
info['position'] * obs[3]

In [ ]:
94.5273631840796 * 100.55 + 485.74524999999994

In [ ]:
(9990.471618159203 / 10_000 - 1)

In [ ]:
env.get_trade_history()

In [ ]:
(9972.511419154229 / 10_000 - 1)

In [ ]:
obs, reward, terminated, truncated, info = env.step(action=Action.SELL)

In [ ]:
env.get_trade_history()

In [ ]:
obs

In [ ]:
info

In [ ]:
from stable_baselines3.common.vec_env import VecFrameStack
vec_env = VecFrameStack(env, n_stack=4)

In [ ]:
from stable_baselines3.common.monitor import Monitor

In [ ]:
m_vec = Monitor(env, filename='monitor.csv')

In [ ]:
m_vec.reset()

In [ ]:
def run_environment_random(make_env, n_steps=1000):
    _env = Monitor(make_env(), filename='monitor.csv')
    _env.reset()
    for _ in range(n_steps):
        action = _env.action_space.sample()
        obs, reward, terminated, truncated, info = _env.step(action)
        if terminated or truncated:
            _env.reset()

    return _env.get_episode_times(), _env.get_episode_rewards(), _env.get_episode_lengths()

In [ ]:
t, r, l = run_environment_random(make_env=make_env, n_steps=1000)

In [ ]:
from statistics import mean, stdev
mean(r), stdev(r)

In [ ]:
mean(l), stdev(l)

In [ ]:
# episodes = []
# for _ in range(100):
#     actions = []
#     rewards = []
#     action = m_vec.action_space.sample()
#     obs, reward, terminated, truncated, info = m_vec.step(action)
#
#     actions.append(action)
#     rewards.append(reward)
#
#     if terminated or truncated:
#         m_vec.reset()
#         episodes.append({'actions': actions, 'rewards': rewards})

In [ ]:
from stable_baselines3 import A2C

In [ ]:
model = A2C("MlpPolicy", make_env(), verbose=1)

In [ ]:
model.learn(total_timesteps=100_000)

In [ ]:
def run_environment_model(make_env, model, n_steps=1000):
    _env = Monitor(make_env(), filename='monitor.csv')
    obs, info = _env.reset()
    for _ in range(n_steps):
        action, _ = model.predict(obs)
        obs, reward, terminated, truncated, info = _env.step(action)
        if terminated or truncated:
            obs, info = _env.reset()

    return _env.get_episode_times(), _env.get_episode_rewards(), _env.get_episode_lengths()

In [ ]:
t, r, l = run_environment_model(make_env=make_env, model=model, n_steps=1000)

In [ ]:
t, r, l

In [ ]:
mean(r), stdev(r)

In [ ]:
mean(l), stdev(l)

In [ ]:
test_env = Monitor(make_env(), filename='monitor_test.csv')
obs, info = test_env.reset()
episodes = []
actions = []
rewards = []
for _ in range(100):

    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, info = test_env.step(action)

    actions.append(action)
    rewards.append(reward)

    if terminated or truncated:
        test_env.reset()
        episodes.append({'actions': actions.copy(), 'rewards': rewards.copy()})
        actions = []
        rewards = []

In [ ]:
episodes

In [ ]:
m_vec.get_total_steps()

In [ ]:
test_env.get_total_steps()

In [ ]:
test_env.get_episode_rewards()

In [ ]:
test_env.get_episode_lengths()

## Use Monitor Wrapper

In [ ]:
env = Monitor(make_env())
action = env.action_space.sample()

In [ ]:
action

In [ ]:
obs, info = env.reset()
env.step(action)

In [ ]:
model.predict(obs)

## Use VecNormalize

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

In [ ]:
def make_monitored_env():
    return Monitor(make_env(), filename='monitor.csv')

In [ ]:
env = VecNormalize(DummyVecEnv([make_monitored_env]))

In [ ]:
env

In [ ]:
obs = env.reset()
for _ in range(100):
    action = env.action_space.sample()
    # obs, reward, terminated, truncated, info = env.step(action)
    obs, reward, done, info = env.step([action])
    if terminated or truncated:
        env.reset()


In [ ]:
env.get_original_reward()

In [ ]:
env.env_method('get_episode_rewards')

In [ ]:
env.env_method('get_episode_lengths')

In [ ]:
model = A2C(
    "MlpPolicy",
    env=VecNormalize(DummyVecEnv([make_monitored_env]), ),
    verbose=1,
    tensorboard_log="./tb20/"
)
model.learn(total_timesteps=100_000)

In [ ]:
model = A2C(
    "MlpPolicy",
    env=make_monitored_env(),
    verbose=1,
    tensorboard_log="./tb20/"
)
model.learn(total_timesteps=200_000)

In [ ]:
t, r, l = run_environment_random(make_env=make_env, n_steps=1000)

In [ ]:
mean(r), stdev(r)

In [ ]:
mean(l), stdev(l)